In [1]:
import os

base = '/kaggle/input/datasets'
for owner in os.listdir(base):
    owner_path = os.path.join(base, owner)
    print(f"{owner}/")
    if os.path.isdir(owner_path):
        for dataset in os.listdir(owner_path):
            print(f"  {dataset}/")

kmader/
  skin-cancer-mnist-ham10000/
nirajankunwor/
  scin-dataset/
  ddi-dataset/
andrewmvd/
  isic-2019/


In [2]:
import pandas as pd
import os
import ast

# ============================================================
# PATHS
# ============================================================
paths = {
    "ham10000": "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000",
    "isic2019": "/kaggle/input/datasets/andrewmvd/isic-2019",
    "ddi": "/kaggle/input/datasets/nirajankunwor/ddi-dataset",
    "scin": "/kaggle/input/datasets/nirajankunwor/scin-dataset",
}

# ============================================================
# HAM10000 (unchanged, already correct)
# ============================================================
ham_meta = pd.read_csv(f"{paths['ham10000']}/HAM10000_metadata.csv")

def ham_image_path(image_id):
    for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        p = f"{paths['ham10000']}/{folder}/{image_id}.jpg"
        if os.path.exists(p):
            return p
    return None

ham_unified = pd.DataFrame({
    "image_path": ham_meta["image_id"].apply(ham_image_path),
    "label": ham_meta["dx"],
    "patient_id": ham_meta["lesion_id"],
    "source_dataset": "ham10000"
})

print("HAM10000:", ham_unified.shape, "| missing images:", ham_unified["image_path"].isna().sum())

# ============================================================
# ISIC 2019 — FIXED: extra nested folder
# ============================================================
isic_labels = pd.read_csv(f"{paths['isic2019']}/ISIC_2019_Training_GroundTruth.csv")
isic_meta = pd.read_csv(f"{paths['isic2019']}/ISIC_2019_Training_Metadata.csv")

label_cols = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]
isic_labels["label"] = isic_labels[label_cols].idxmax(axis=1)

isic_merged = isic_labels.merge(isic_meta, on="image", how="left")

isic_unified = pd.DataFrame({
    "image_path": isic_merged["image"].apply(
        lambda x: f"{paths['isic2019']}/ISIC_2019_Training_Input/ISIC_2019_Training_Input/{x}.jpg"
    ),
    "label": isic_merged["label"],
    "patient_id": isic_merged["lesion_id"].fillna(isic_merged["image"]),
    "source_dataset": "isic2019"
})

print("ISIC2019:", isic_unified.shape, "| missing images:", isic_unified["image_path"].apply(lambda p: not os.path.exists(p)).sum())

# ============================================================
# DDI (unchanged, already correct)
# ============================================================
ddi_meta = pd.read_csv(f"{paths['ddi']}/ddi_metadata.csv")

ddi_unified = pd.DataFrame({
    "image_path": ddi_meta["DDI_file"].apply(lambda x: f"{paths['ddi']}/images/{x}"),
    "label": ddi_meta["disease"],
    "malignant": ddi_meta["malignant"],
    "skin_tone": ddi_meta["skin_tone"],
    "patient_id": ddi_meta["DDI_ID"],
    "source_dataset": "ddi"
})

print("DDI:", ddi_unified.shape, "| missing images:", ddi_unified["image_path"].apply(lambda p: not os.path.exists(p)).sum())

# ============================================================
# SCIN — FIXED: strip the 'dataset/images/' prefix already in the CSV
# ============================================================
scin_cases = pd.read_csv(f"{paths['scin']}/scin_cases.csv")
scin_labels = pd.read_csv(f"{paths['scin']}/scin_labels.csv")
scin_merged = scin_cases.merge(scin_labels, on="case_id", how="inner")

def parse_top_label(label_str):
    try:
        label_dict = ast.literal_eval(label_str)
        if not label_dict:
            return None
        return max(label_dict, key=label_dict.get)
    except (ValueError, SyntaxError, TypeError):
        return None

scin_merged["parsed_label"] = scin_merged["weighted_skin_condition_label"].apply(parse_top_label)

n_before = len(scin_merged)
scin_merged_clean = scin_merged[scin_merged["parsed_label"].notna()].copy()
n_after = len(scin_merged_clean)

print(f"\nSCIN cases before filtering: {n_before}")
print(f"SCIN cases after dropping empty labels: {n_after} (dropped {n_before - n_after}, {(n_before-n_after)/n_before*100:.1f}%)")

def clean_scin_path(raw_path):
    # raw_path looks like "dataset/images/-3205742176803893704.png"
    # actual files sit directly at {scin_path}/images/<filename>
    filename = os.path.basename(raw_path)
    return f"{paths['scin']}/images/{filename}"

scin_rows = []
for _, row in scin_merged_clean.iterrows():
    for img_col in ["image_1_path", "image_2_path", "image_3_path"]:
        img_path = row.get(img_col)
        if pd.notna(img_path):
            scin_rows.append({
                "image_path": clean_scin_path(img_path),
                "label": row["parsed_label"],
                "fitzpatrick_skin_type": row["fitzpatrick_skin_type"],
                "patient_id": row["case_id"],
                "source_dataset": "scin"
            })

scin_unified = pd.DataFrame(scin_rows)

print("SCIN unified:", scin_unified.shape, "| missing images:", scin_unified["image_path"].apply(lambda p: not os.path.exists(p)).sum())
print("\nTop SCIN labels after cleaning:")
print(scin_unified["label"].value_counts().head(15))

# ============================================================
# COMBINE + FINAL SUMMARY
# ============================================================
all_datasets = {
    "ham10000": ham_unified,
    "isic2019": isic_unified,
    "ddi": ddi_unified,
    "scin": scin_unified,
}

print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
for name, df in all_datasets.items():
    print(f"{name:12s} rows={len(df):6d}  unique_patients={df['patient_id'].nunique():6d}  unique_labels={df['label'].nunique():3d}")

os.makedirs("/kaggle/working/unified", exist_ok=True)
for name, df in all_datasets.items():
    df.to_csv(f"/kaggle/working/unified/{name}_unified.csv", index=False)

print("\nSaved unified tables to /kaggle/working/unified/")

HAM10000: (10015, 4) | missing images: 0
ISIC2019: (25331, 4) | missing images: 0
DDI: (656, 6) | missing images: 0

SCIN cases before filtering: 5033
SCIN cases after dropping empty labels: 3061 (dropped 1972, 39.2%)
SCIN unified: (6518, 5) | missing images: 1

Top SCIN labels after cleaning:
label
Eczema                         1079
Allergic Contact Dermatitis     590
Urticaria                       442
Insect Bite                     401
Folliculitis                    306
Psoriasis                       234
Tinea                           212
Impetigo                        136
Herpes Zoster                   130
Drug Rash                       129
Pigmented purpuric eruption     128
Acne                            123
Herpes Simplex                  109
CD - Contact dermatitis          98
Acute dermatitis, NOS            93
Name: count, dtype: int64

FINAL SUMMARY
ham10000     rows= 10015  unique_patients=  7470  unique_labels=  7
isic2019     rows= 25331  unique_patients= 13931  

In [3]:
print(ham_unified.shape, isic_unified.shape)

(10015, 4) (25331, 4)


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score
import os, json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 1. MAP HAM10000 + ISIC TO A SHARED 8-CLASS TAXONOMY
# ============================================================
ham_map = {
    "mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AK",
    "bkl": "BKL", "df": "DF", "vasc": "VASC"
}
isic_map = {
    "MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AK",
    "BKL": "BKL", "DF": "DF", "VASC": "VASC", "SCC": "SCC", "UNK": None
}

ham = ham_unified.copy()
ham["mapped_label"] = ham["label"].map(ham_map)

isic = isic_unified.copy()
isic["mapped_label"] = isic["label"].map(isic_map)

combined = pd.concat([
    ham[["image_path", "mapped_label", "patient_id", "source_dataset"]],
    isic[["image_path", "mapped_label", "patient_id", "source_dataset"]]
], ignore_index=True)
combined = combined[combined["mapped_label"].notna()].reset_index(drop=True)

classes = sorted(combined["mapped_label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
combined["y"] = combined["mapped_label"].map(class_to_idx)

print("Classes:", classes)
print("Combined training pool:", combined.shape)
print(combined["mapped_label"].value_counts())

# ============================================================
# 2. PATIENT-LEVEL TRAIN/VAL/TEST SPLIT (no patient leakage)
# ============================================================
combined["pid"] = combined["source_dataset"] + "_" + combined["patient_id"].astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss.split(combined, groups=combined["pid"]))
train_df = combined.iloc[train_idx].reset_index(drop=True)
temp_df = combined.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["pid"]))
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df["pid"]) & set(val_df["pid"]))
assert not (set(train_df["pid"]) & set(test_df["pid"]))
assert not (set(val_df["pid"]) & set(test_df["pid"]))
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}  (patient-level, no leakage)")

# ============================================================
# 3. DATASET + DATALOADERS
# ============================================================
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        return self.tf(img), row["y"]

train_loader = DataLoader(SkinDataset(train_df, train_tf), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(SkinDataset(val_df, eval_tf), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SkinDataset(test_df, eval_tf), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# 4. MODEL: ResNet-50, class-weighted loss
# ============================================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

counts = train_df["y"].value_counts().sort_index().values
weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

# ============================================================
# 5. TRAIN — with resumable checkpointing
# ============================================================
EPOCHS = 15
ckpt_path = "/kaggle/working/checkpoint.pt"
best_path = "/kaggle/working/baseline_resnet50_best.pt"

start_epoch = 0
best_val_bacc = 0.0

# Resume if a checkpoint exists (survives session restart)
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_bacc = ckpt["best_val_bacc"]
    print(f"Resumed from checkpoint at epoch {start_epoch}, best_val_bacc so far {best_val_bacc:.3f}")
else:
    print("No checkpoint found, starting fresh.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running = 0.0
    for imgs, ys in train_loader:
        imgs, ys = imgs.to(device), ys.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, ys)
        loss.backward()
        optimizer.step()
        running += loss.item()
    scheduler.step()

    # validation
    model.eval()
    vy, vp = [], []
    with torch.no_grad():
        for imgs, ys in val_loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu().numpy()
            vp.extend(preds); vy.extend(ys.numpy())
    val_bacc = balanced_accuracy_score(vy, vp)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train_loss={running/len(train_loader):.3f} | val_balanced_acc={val_bacc:.3f}")

    # save best model
    if val_bacc > best_val_bacc:
        best_val_bacc = val_bacc
        torch.save(model.state_dict(), best_path)

    # save resumable checkpoint EVERY epoch (overwrites — survives session death)
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_bacc": best_val_bacc,
    }, ckpt_path)

print(f"\nBest val balanced accuracy: {best_val_bacc:.3f}")

# ============================================================
# 6. SAVE SPLITS + CLASS MAPPING (reused by eval + feature extraction)
# ============================================================
os.makedirs("/kaggle/working/unified", exist_ok=True)
train_df.to_csv("/kaggle/working/unified/baseline_train.csv", index=False)
val_df.to_csv("/kaggle/working/unified/baseline_val.csv", index=False)
test_df.to_csv("/kaggle/working/unified/baseline_test.csv", index=False)
with open("/kaggle/working/unified/class_mapping.json", "w") as f:
    json.dump(class_to_idx, f)
print("Saved model, splits, and class mapping to /kaggle/working/")

Device: cuda
Classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']
Combined training pool: (35346, 5)
mapped_label
NV      19580
MEL      5635
BCC      3837
BKL      3723
AK       1194
SCC       628
VASC      395
DF        354
Name: count, dtype: int64

Train: 24687 | Val: 5281 | Test: 5378  (patient-level, no leakage)
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 148MB/s]


No checkpoint found, starting fresh.
Epoch  1/15 | train_loss=1.257 | val_balanced_acc=0.622
Epoch  2/15 | train_loss=0.750 | val_balanced_acc=0.649
Epoch  3/15 | train_loss=0.579 | val_balanced_acc=0.639
Epoch  4/15 | train_loss=0.425 | val_balanced_acc=0.656
Epoch  5/15 | train_loss=0.323 | val_balanced_acc=0.671
Epoch  6/15 | train_loss=0.247 | val_balanced_acc=0.648
Epoch  7/15 | train_loss=0.190 | val_balanced_acc=0.642
Epoch  8/15 | train_loss=0.148 | val_balanced_acc=0.645
Epoch  9/15 | train_loss=0.120 | val_balanced_acc=0.673
Epoch 10/15 | train_loss=0.093 | val_balanced_acc=0.666
Epoch 11/15 | train_loss=0.081 | val_balanced_acc=0.665
Epoch 12/15 | train_loss=0.065 | val_balanced_acc=0.651
Epoch 13/15 | train_loss=0.061 | val_balanced_acc=0.660
Epoch 14/15 | train_loss=0.055 | val_balanced_acc=0.655
Epoch 15/15 | train_loss=0.054 | val_balanced_acc=0.673

Best val balanced accuracy: 0.673
Saved model, splits, and class mapping to /kaggle/working/
